# Tearing a blood-clot sheet: a simulation-based measurement of its toughness

This notebook replicates, with the `mime-softtissue` plugin for MIME, a published fracture finding on human whole-blood clot:
the notched **pure-shear** tests of Lohr, Bechtel, Dortdivanlioglu and Rausch (2025), *How to measure fracture toughness of
soft materials: a comparison of six different approaches using blood clot as a model material*, Int. J. Fract.,
doi:10.1007/s10704-024-00820-4. Their six methods gave a toughness of 0.2 to 2.9 J/m² (mean 0.9); the same lab reported
3.7 ± 0.88 J/m² for human clot by another protocol (Sugerman et al. 2024, J. Mech. Behav. Biomed. Mater. 154, 106508).
Because the published numbers disagree by a factor of four, this is a **seventh method**, judged against the measured curves:

1. the sheet's stiffness (a Yeoh law) is fitted to the unnotched curve;
2. the toughness Gc is taken from one notch group's peak force (group 4, mean notch 11.9 mm);
3. the other groups' force against displacement, peak forces and crack-start displacements are **predictions**.

**The data and their terms.** The authors' raw data, CC0 1.0, shipped in the plugin as two records: the 15
force-displacement files (Texas Data Repository, doi:10.18738/T8/97ZAZN; `softtissue.lohr2025_pure_shear`) and each notched
sample's crack-start displacement, read from the DIC images (doi:10.18738/T8/DUJWSR;
`softtissue.lohr2025_critical_displacements`). The means and ±1 SD bands below are computed from the files. Each sample's
notch length is the only number taken from the paper's figures (Fig. 5A, digitised): the files carry nominal sizes only.
The files show a grip rate of 0.2 mm/s; the paper's text says 0.1 mm/s.

**The model:** a 2-D incompressible plane-stress half sheet (40 × 5 mm, 3 mm thick, symmetric about the crack plane), the
Yeoh law inside an AT2 phase field (length scale 0.25 mm), the notch a released part of the symmetry plane, the grips
clamped. Pure JAX, differentiable in forward mode.

In [ ]:
# Where this runs: Colab or a local Jupyter. The flags are set before JAX is imported.
import os, sys
in_colab = "google.colab" in sys.modules
os.environ.setdefault("XLA_FLAGS", "--xla_gpu_enable_triton_gemm=false")
os.environ.setdefault("JAX_DEFAULT_MATMUL_PRECISION", "highest")
os.environ.setdefault("JAX_ENABLE_X64", "1")
print("in Colab:", in_colab)

## Install (Colab only)

MIME from its public release branch, then the plugin from its **private** repository with a GitHub token kept in Colab's
secrets under the name `GH_PAT` (the key icon on the left, *Add new secret*, and give this notebook access). The token goes
into the install command only; it is never printed, logged or written to a file, and pip's output is not shown.

This notebook has been executed end to end outside Colab (a clean local environment, with this install cell skipped
by the `in_colab` check); the install cell itself has not been run on Colab yet.

In [ ]:
# The plugin's branch that carries this study on the raw data (not merged; "main" once it is).
PLUGIN_REF = "raw-data"
if in_colab:
    import subprocess
    from google.colab import userdata

    def pip_install(spec, secret=None, *options):
        done = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *options, spec], capture_output=True,
                              text=True)
        if done.returncode:
            message = done.stderr if secret is None else done.stderr.replace(secret, "***")
            raise RuntimeError("pip failed:\n" + message[-2000:])

    pip_install("mime-engine[verify] @ git+https://github.com/Microrobotics-Simulation-Framework/MIME@release/v0.3.0")
    _token = userdata.get("GH_PAT")
    pip_install("git+https://" + _token + "@github.com/Microrobotics-Simulation-Framework/mime-softtissue.git@"
                + PLUGIN_REF, _token, "--no-cache-dir")
    del _token
    print("installed")

In [ ]:
# CPU or GPU, by what the runtime gives: JAX takes a GPU when there is one (full matmul precision and no Triton
# GEMM, set above), the CPU otherwise. The engine runs in float64 either way.
import jax
print("JAX devices:", jax.devices())

import json
import time
import numpy as np
import matplotlib.pyplot as plt
from mime_softtissue.studies import pure_shear as ps
from mime_softtissue.studies import results as published

D = ps.data()
R = published.load()
print(D["source"])

## The data (the authors' files)

The three unnotched curves' mean ± 1 SD, and the five notch groups' mean curves, from the raw files. The groups are the
paper's (mean notches 3.7, 7.3, 9.7, 11.9 and 16.5 mm) and hold 3, 2, 1, 3 and 3 samples.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
u = np.array(D["unnotched"]["mean"]); b = D["unnotched"]["band"]
ax[0].fill_between(b["x"], b["lower"], b["upper"], color="0.85", label="±1 SD [measured]")
ax[0].plot(u[:, 0], u[:, 1], "k", label="unnotched mean [measured]")
ax[0].set_title("unnotched (3 samples)")
colors = plt.cm.viridis(np.linspace(0, 0.85, 5))
for g in "12345":
    c = np.array(D["group_curves"][g]["points"])
    ax[1].plot(c[:, 0], c[:, 1], color=colors[int(g) - 1],
               label=f"group {g}: notch {D['groups'][g]['notch_mean_mm']:.1f} mm, n = {D['groups'][g]['n']}")
ax[1].set_title("notched, group means [measured]")
for a in ax:
    a.set_xlabel("grip displacement (mm)"); a.legend(fontsize=8)
ax[0].set_ylabel("force (N)")
plt.tight_layout(); plt.show()

## A coarse case, live

The calibrated model (its constants from the converged runs below) on a coarse grid, h = 0.25 mm against the converged
runs' 0.0625 mm: group 4's sheet pulled to just past its peak. A few minutes on a laptop or Colab CPU; coarse numbers.

In [ ]:
C = R["calibrated"]
k = ps.Stiffness(C["c1"], C["c2"])
notch = D["groups"]["4"]["notch_mean_mm"] * 1e-3
t0 = time.time()
live = ps.run_case(notch, k, C["gc"], C["l_mm"] * 1e-3, 0.25e-3, delta_max_mm=5.0, step_mm=0.1, keep_every=1,
                   stop_below=0.6)
fp, dp = ps.peak(live)
dc = ps.critical_displacement(live)
print(f"coarse group 4: peak {fp * 1e3:.1f} mN at {dp:.2f} mm, crack start "
      f"{'none' if dc is None else f'{dc:.2f} mm'} ({time.time() - t0:.0f} s)")

## The converged runs against the data

Loaded from the plugin's `studies/results/` (computed on GitHub Actions at h = 0.0625 mm). Group 4 is the calibration;
every other curve is a prediction. The bands: ±1 SD across each group's samples at its mean curve's peak, and the
measured crack-start displacement's mean ± 1 SD. The runs were made at Gc = 0.819 J/m² on the digitised curves; on the raw
files the calibration gives 0.826, which moves these curves by under 0.5 %. Each model curve is drawn up to the load step at which its crack has
run through the sheet: past it the model's fully damaged row on the crack plane, stretched 25 to 48 times, still carries a
rising force through its residual stiffness, an artefact that is not shown (`R["after_separation"]`).

In [ ]:
print(json.dumps(C, indent=1))
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
for g in "12345":
    c = np.array(D["group_curves"][g]["points"]); col = colors[int(g) - 1]
    ax[0].plot(c[:, 0], c[:, 1], "--", color=col)
    m = R["groups"][g]
    cut = int(np.searchsorted(m["delta_mm"], m["separated_at_mm"] + 1e-9)) if m["separated_at_mm"] else None
    ax[0].plot(m["delta_mm"][:cut], m["force_N"][:cut], color=col,
               label=f"group {g}" + (" (calibration)" if g == str(C["group"]) else ""))
ax[0].plot(live["delta_mm"], live["force_N"], ":", color="k", label="group 4, live, coarse")
ax[0].set_xlabel("grip displacement (mm)"); ax[0].set_ylabel("force (N)"); ax[0].legend(fontsize=8)
ax[0].set_title("model (solid) and data means (dashed, measured)")
for g in "12345":
    dg, m, cv, col = D["groups"][g], R["groups"][g], D["group_curves"][g], colors[int(g) - 1]
    a = dg["notch_mean_mm"]
    ax[1].errorbar(a - 0.2, cv["peak_N"], yerr=cv["peak_sd_N"] or 0.0, fmt="s", color=col,
                   mfc="none")
    ax[1].plot(a + 0.2, m["peak_N"], "o", color=col)
    ax[2].errorbar(a - 0.2, dg["delta_c_mean_mm"], yerr=dg["delta_c_sd_mm"] or 0.0, fmt="s", color=col, mfc="none")
    if m["delta_c_mm"] is not None:
        ax[2].plot(a + 0.2, m["delta_c_mm"], "o", color=col)
ax[1].set_xlabel("notch length (mm)"); ax[1].set_ylabel("peak force (N)")
ax[1].set_title("peak force: data (open) and model (filled)")
ax[2].set_xlabel("notch length (mm)"); ax[2].set_ylabel("crack-start displacement (mm)")
ax[2].set_title("crack start: data (DIC, open) and model (filled)")
plt.tight_layout(); plt.show()
print("group  notch   peak force data / model (N)   crack start data / model (mm)")
for g in "12345":
    dg, m = D["groups"][g], R["groups"][g]
    sd = dg["delta_c_sd_mm"]
    band = f"+- {sd:.2f}" if sd is not None else "(n = 1)"
    model = f"{m['delta_c_mm']:.2f}" if m["delta_c_mm"] is not None else "none"
    print(f"  {g}   {dg['notch_mean_mm']:5.2f}   {D['group_curves'][g]['peak_N']:.4f} / {m['peak_N']:.4f}"
          f"            {dg['delta_c_mean_mm']:.2f} {band} / {model}")

## What the comparison shows

The numbers below are this study's (h = 0.0625 mm, l = 0.25 mm), on the raw files. **Stiffness:** the Yeoh fit to the
unnotched mean holds to 1.4 % of its maximum (NRMSE) at the calibrated toughness. **Toughness:** Gc = 0.83 ± 0.17 J/m²
from group 4's peak force alone (± over leaving one sample out), inside the authors' Griffith-A mean of
0.9 ± 0.38 J/m². **Held-out predictions:** with one Gc, group 5's peak force lies in its band (58.3 against
57.2 ± 5.6 mN) but groups 1 and 2 fall 24 % low (94.8 against 124.4 ± 13.7 mN, 81.1 against 106.6 ± 9.4 mN); the
crack-start displacement lies in its band for groups 1 and 5, not 2. **A single toughness does not carry across the notch
lengths:** the Gc each group's own peak needs falls with the notch, about 1.55 J/m² for the 3.7 and 7.3 mm notches, 1.35
for 9.7 mm (one sample), 0.83 and 0.78 for 11.9 and 16.5 mm (interpolated in a scan of Gc), the direction the authors'
Fig. 5 shows. The model's force also peaks earlier (2.9 to 3.06 mm against the data's 3.7 to 4.84 mm) and drops at once,
while the measured crack start lies 1 to 2 mm of grip displacement before each measured peak: the measured cracks grow
stably before the peak, the model's do not. The
study's report gives every check (grid, energy balance, planted errors).

## The animation

**A replay of the recorded frames, not a new simulation.** The sheet tearing, coloured by the phase field's damage (0 to 1):
the frames saved from group 4's converged run (h = 0.0625 mm) are replayed through MIME as a deforming surface
(`softtissue.replayed_tear`, which declares a `SurfaceMesh` with a damage `MeshField`), recorded by an `Experiment`, and
shown with MIME's own page. Both halves of the sheet are drawn, the crack plane opened; the front face is displaced by the
sheet's thinning for display only. The page is coarser than the run (every 8th grid column, every second saved frame, to
3.6 mm of grip displacement, just past the crack's run through the sheet); its clock is the test's, 0.2 mm/s. Press play
and drag to turn the sheet.

**Where the replay's frames come from.** `ReplayedTear` reads a file shipped inside the plugin, `mime_softtissue/studies/results/pure_shear_frames.npz`: the group-4 run's saved displacement and damage fields, written by the converged run above. The entry does not declare that file to MIME yet, so the saved experiment's lock does not pin it. A later version passes it as a declared record (`frames=asset("softtissue.pure_shear_tear_g4")`), so the data a run depends on is named, pinned by its sha256 and carries its licence.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp

from mime.experiment import Experiment
from mime.scene import Scene, Z
from mime.things import Body, State
import mime.units as mu
from mime.viz import display
from mime_softtissue.entries import ReplayedTear

replayed = ReplayedTear(every_column=8, every_frame=2)  # a replay of saved frames: it runs nothing
tear = Experiment("pure_shear_tear", scene=Scene.new(up=Z, units="mm"), gravity=mu.m_per_s2(0.0),
                  dt=mu.s(replayed.step_seconds))
tear.add_body(Body("sheet", dynamics=replayed, initial=State(position=mu.m((0.0, 0.0, 0.0)))))
folder = Path(mkdtemp()) / "tear"
tear.build().run_to(folder, mu.s(replayed.duration_seconds), stage_every=mu.s(replayed.step_seconds), history=True,
                      progress=False)
display(folder, caption=f"Replay of the recorded frames of group 4's converged run (notch "
        f"{D['groups']['4']['notch_mean_mm']:.1f} mm, h = {C['h_mm']} mm), not a new simulation: both halves of the "
        f"sheet, the crack plane opened. Colour: the phase field's damage, 0 to 1. The front face is displaced by the "
        f"sheet's thinning for display only. Every 8th grid column and every second saved frame, to 3.6 mm of grip "
        f"displacement; time is the test's clock at 0.2 mm/s.")

## What the model lacks

- **Viscoelasticity.** Clot relaxes (time constants of about 3 s and 61 s) and keeps a set of about 16 % after cyclic
  loading (Sugerman et al. 2024 measured both). The model is elastic; the tests ran at 0.2 mm/s, so rate effects are in the
  data and not in the model.
- **The stiffening law is a refit.** The authors fitted a one-term Ogden law to each notched sample; here a two-term Yeoh
  law is fitted to the unnotched mean (an Ogden law's derivatives misbehave at the undeformed state in this engine's
  invariant form).
- **Plane stress is approximated.** The 3 mm sheet is a 2-D incompressible plane-stress sheet, checked against a thin 3-D
  slab on one elastic case; the clot's fluid (about half its volume) and any fluid loss are ignored.
- **One toughness and one stiffness for every sample.** The notched samples' own stiffness is not fitted; a single Gc
  is the hypothesis the predictions test.
- **Notch lengths read from a figure.** Each sample's notch length is digitised from Fig. 5A (the files carry nominal
  sizes); everything else is the authors' raw data.